<a href="https://colab.research.google.com/github/SuchandanG/Optimized-Secure-Admissible-Partitioning-of-Affine-Manifolds-for-General-Access-Structures/blob/main/Secret_sharing_with_optimal_SAP_using_PAD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Secret Sharing using optimal Secure-Admissible Partitioning satisfying PAD

##Optimal SAP satisfying Parity Absorption Duality

In [ ]:
from itertools import combinations


# =========================================================
# Parse input sets
# =========================================================

def parse_set(s):

    s = s.strip()

    s = s.replace("{", "").replace("}", "")

    if s == "":

        return frozenset()

    return frozenset(map(int, s.split(",")))


# =========================================================
# Local intersection
# =========================================================

def local_intersection(subfamily):

    return set.intersection(*map(set, subfamily))


# =========================================================
# Quotient residuals
# =========================================================

def quotient_residuals(subfamily, S):

    return [set(A) - S for A in subfamily]


# =========================================================
# k-SC condition
# =========================================================

def satisfies_kSC(residuals):

    k = len(residuals)

    for i in range(k):

        ok = True

        for j in range(k):

            if i == j:

                continue

            if residuals[i].intersection(residuals[j]):

                ok = False

                break

        if ok:

            return True

    return False


# =========================================================
# XOR-induced support over F2
# =========================================================

def xor_support(subfamily):

    count = {}

    for A in subfamily:

        for x in A:

            count[x] = count.get(x, 0) + 1

    B = set()

    for x in count:

        if count[x] % 2 == 1:

            B.add(x)

    return frozenset(B)


# =========================================================
# PARITY CLOSURE OF A BLOCK
# =========================================================

def closure_sets(block):

    derived = set()

    n = len(block)

    for k in range(3, n + 1, 2):

        for sub in combinations(block, k):

            X = xor_support(sub)

            if X not in block:
                derived.add(X)

    return derived


# =========================================================
# MONOTONE QUALIFICATION CHECK
# =========================================================

def monotone_qualified(B, Gamma_0):

    for A in Gamma_0:

        if set(A).issubset(set(B)):

            return True

    return False


# =========================================================
# PROPERTY A CHECK
# =========================================================

def admissible_family(family, Gamma_0):

    n = len(family)

    for k in range(3, n + 1, 2):

        for sub in combinations(family, k):

            S = local_intersection(sub)

            residuals = quotient_residuals(sub, S)

            # -------------------------------------------------
            # Property A failure
            # -------------------------------------------------

            if not satisfies_kSC(residuals):

                B = xor_support(sub)

                # -------------------------------------------------
                # Property B admissibility
                # -------------------------------------------------

                if not monotone_qualified(B, Gamma_0):

                    return False

    return True


# =========================================================
# CHECK WHETHER B IS GENERATED
# BY CURRENT FAMILY
#
# IMPORTANT:
# B must be generated by OTHER sets
# =========================================================

def generated_by_family(B, family):

    n = len(family)

    for k in range(3, n + 1, 2):

        for sub in combinations(family, k):

            X = xor_support(sub)

            if X == B:

                return True

    return False


# =========================================================
# MINIMAL AFFINE GENERATING BLOCK
#
# Greedy basis extraction
# =========================================================

def reduce_block(block):

    generators = list(block)

    derived = set()

    changed = True

    while changed:

        changed = False

        # -------------------------------------------------
        # try removing one dependent set
        # -------------------------------------------------

        for B in generators.copy():

            others = [
                A for A in generators
                if A != B
            ]

            # -------------------------------------------------
            # generated by others
            # -------------------------------------------------

            if generated_by_family(B, others):

                generators.remove(B)

                derived.add(B)

                changed = True

                break

    return generators, derived


# =========================================================
# SUPPORT OF BLOCK
# =========================================================

def block_support(block):

    S = set()

    for A in block:

        S |= set(A)

    return S


# =========================================================
# GLOBAL SHARE EXPANSION FUNCTIONAL
#
# C(Pi)=sum_i |supp(Gamma_i)|
# =========================================================

def share_expansion_cost(partition):

    total = 0

    for block in partition:

        total += len(block_support(block))

    return total


# =========================================================
# FORMAT BLOCK
# =========================================================

def format_block(block):

    formatted = []

    for A in sorted(block, key=lambda x: sorted(list(x))):

        formatted.append(
            "{" + ",".join(map(str, sorted(A))) + "}"
        )

    return formatted


# =========================================================
# GENERATE ALL SET PARTITIONS
# =========================================================

def all_partitions(collection):

    if len(collection) == 1:

        yield [collection]

        return

    first = collection[0]

    for smaller in all_partitions(collection[1:]):

        # -------------------------------------------------
        # insert into existing block
        # -------------------------------------------------

        for n, subset in enumerate(smaller):

            yield (
                smaller[:n]
                + [[first] + subset]
                + smaller[n + 1:]
            )

        # -------------------------------------------------
        # create singleton block
        # -------------------------------------------------

        yield [[first]] + smaller


# =========================================================
# CHECK GLOBAL SAP PARTITION
# =========================================================

def admissible_partition(partition, Gamma_0):

    reduced_partition = []
    derived_per_block = []

    for block in partition:

        if not admissible_family(block, Gamma_0):
            return False, [], set()


        reduced, _ = reduce_block(block)

        derived = closure_sets(reduced)

        reduced_partition.append(reduced)
        derived_per_block.append(derived)

# --------------------------------------------
# NEW CHECK
# Derived sets cannot appear in another block
# --------------------------------------------

    for i in range(len(reduced_partition)):

        for j in range(len(reduced_partition)):

            if i == j:
                continue

            for D in derived_per_block[i]:

                if D in reduced_partition[j]:
                    return False, [], set()

    all_derived = set().union(*derived_per_block)

    return (
    True,
    reduced_partition,
    all_derived
    )


# =========================================================
# GLOBAL SAP OPTIMIZATION
# =========================================================

def globally_optimal_partition(Gamma_input):

    Gamma_0 = set(Gamma_input)

    best_partition = None

    best_derived = set()

    best_cost = None

    checked = 0

    admissible_count = 0

    # -----------------------------------------------------
    # enumerate ALL partitions
    # -----------------------------------------------------

    for partition in all_partitions(list(Gamma_0)):

        checked += 1

        (
            admissible,
            reduced_partition,
            derived_sets
        ) = admissible_partition(
            partition,
            Gamma_0
        )

        if admissible:

            admissible_count += 1

            cost = share_expansion_cost(
                reduced_partition
            )

            # -------------------------------------------------
            # global minimization
            # -------------------------------------------------

            if (
                best_cost is None
                or cost < best_cost
            ):

                best_cost = cost

                best_partition = reduced_partition

                best_derived = derived_sets

    return (
        best_partition,
        best_derived,
        best_cost,
        checked,
        admissible_count
    )


# =========================================================
# DRIVER
# =========================================================

m = int(input("Enter number of minimal qualified sets: "))

Gamma = []

print("\nEnter sets in the form {1,2,3,...,l}\n")

for i in range(m):

    s = input(f"Set {i+1}: ")

    Gamma.append(parse_set(s))


# =========================================================
# RUN
# =========================================================

(
    result,
    derived,
    cost,
    checked,
    admissible_count
) = globally_optimal_partition(Gamma)


# =========================================================
# OUTPUT
# =========================================================

print("\nGlobally Optimal SAP Partition:\n")

for idx, block in enumerate(result, start=1):

    print(
        f"Γ_{idx} =",
        format_block(block)
    )


# =========================================================
# DERIVED CLOSURE
# =========================================================

if derived:

    formatted = []

    for D in sorted(derived, key=lambda x: sorted(list(x))):

        formatted.append(
            "{" + ",".join(map(str, sorted(D))) + "}"
        )

    print("\nParity-Absorbed Derived Sets:")

    print("D(Γ) =", formatted)


# =========================================================
# SUPPORT ANALYSIS
# =========================================================

print("\nPartition Supports:\n")

for idx, block in enumerate(result, start=1):

    supp = sorted(block_support(block))

    print(
        f"supp(Γ_{idx}) =",
        supp,
        f" size = {len(supp)}"
    )


# =========================================================
# SHARE EXPANSION COST
# =========================================================

print("\nGlobally Optimal Share Expansion:\n")

print(
    "C(Pi*) =",
    cost
)


# =========================================================
# COMPLEXITY INFO
# =========================================================

print("\nTotal Partitions Checked =", checked)

print(
    "Total SAP-Admissible Partitions =",
    admissible_count
)


# =========================================================
# THEORETICAL BOUNDS
# =========================================================

all_participants = set()

for A in Gamma:

    all_participants |= set(A)

n = len(all_participants)

num_parts = len(result)

print("\nDistinct Participants =", n)

print("Lower Bound =", n)

print(
    "Upper Bound =",
    num_parts * n
)

Enter number of minimal qualified sets: 6

Enter sets in the form {1,2,3,...,l}

Set 1: {1,2}
Set 2: {1,3}
Set 3: {1,4}
Set 4: {2,3}
Set 5: {2,4}
Set 6: {3,4}

Globally Optimal SAP Partition:

Γ_1 = ['{1,4}', '{2,3}']
Γ_2 = ['{1,2}', '{1,3}', '{2,4}']

Parity-Absorbed Derived Sets:
D(Γ) = ['{3,4}']

Partition Supports:

supp(Γ_1) = [1, 2, 3, 4]  size = 4
supp(Γ_2) = [1, 2, 3, 4]  size = 4

Globally Optimal Share Expansion:

C(Pi*) = 8

Total Partitions Checked = 203
Total SAP-Admissible Partitions = 98

Distinct Participants = 4
Lower Bound = 4
Upper Bound = 8


## ${S}_{\mathrm{GEN}}:$ Share Generation

In [ ]:
import ast
import random

# ==========================================================
# Parse set
# ==========================================================

def parse_set(s):

    s = s.strip()

    s = s.replace("{","")
    s = s.replace("}","")

    if s == "":
        return set()

    return set(map(int,s.split(",")))

# ==========================================================
# Read partition blocks
# ==========================================================

num_partitions = int(input("Enter number of partition blocks: "))

partitions = []

for i in range(num_partitions):

    print(f"\nEnter Γ_{i+1} exactly like")

    print("['{1,2}','{2,3,4}']")

    raw = ast.literal_eval(input("Input : "))

    block = []

    for x in raw:
        block.append(parse_set(x))

    partitions.append(block)

# ==========================================================
# Construct incidence matrix
# ==========================================================

def construct_matrix(block):

    participants = sorted(set().union(*block))

    index = {}

    for i,p in enumerate(participants):
        index[p]=i

    matrix=[]

    for A in block:

        row=[0]*len(participants)

        for x in A:

            row[index[x]]=1

        matrix.append(row)

    return matrix,participants

# ==========================================================
# Gaussian elimination (Rank)
# ==========================================================

def gf2_rank(M):

    A=[row[:] for row in M]

    rows=len(A)

    cols=len(A[0])

    r=0

    c=0

    while r<rows and c<cols:

        pivot=None

        for i in range(r,rows):

            if A[i][c]:

                pivot=i

                break

        if pivot is None:

            c+=1

            continue

        A[r],A[pivot]=A[pivot],A[r]

        for i in range(rows):

            if i!=r and A[i][c]:

                for j in range(c,cols):

                    A[i][j]^=A[r][j]

        r+=1
        c+=1

    return r

# ==========================================================
# Construct all matrices
# ==========================================================

matrices=[]

participant_lists=[]

print("\n==============================")
print("RECONSTRUCTION MATRICES")
print("==============================\n")

for i,block in enumerate(partitions,start=1):

    M,plist=construct_matrix(block)

    matrices.append(M)

    participant_lists.append(plist)

    print(f"Γ_{i}")

    print("Participants :",plist)

    for row in M:

        print(row)

    print("Rank =",gf2_rank(M))

    print()

# ==========================================================
# Secret
# ==========================================================

L=int(input("\nEnter secret length : "))

print(f"Enter {L} bits separated by spaces")

secret_bits=list(map(int,input("Secret : ").split()))

if len(secret_bits)!=L:

    raise ValueError("Incorrect secret length")

for b in secret_bits:

    if b not in [0,1]:

        raise ValueError("Secret must contain only 0 or 1")




# ==========================================================
# Random affine solution over GF(2)
#
# Solves
#
#        M x = (s,...,s)^T
#
# Returns one random solution.
# ==========================================================

def random_affine_solution(M, secret):

    rows = len(M)
    cols = len(M[0])

    # ------------------------------
    # Augmented matrix
    # ------------------------------

    A = []

    for row in M:

        A.append(row[:] + [secret])

    pivot_cols = []

    r = 0

    # ------------------------------
    # Gaussian elimination
    # ------------------------------

    for c in range(cols):

        pivot = None

        for i in range(r, rows):

            if A[i][c]:

                pivot = i
                break

        if pivot is None:
            continue

        A[r], A[pivot] = A[pivot], A[r]

        pivot_cols.append(c)

        # eliminate below

        for i in range(r + 1, rows):

            if A[i][c]:

                for j in range(c, cols + 1):

                    A[i][j] ^= A[r][j]

        r += 1

        if r == rows:
            break

    # ------------------------------
    # Check consistency
    # ------------------------------

    for i in range(r, rows):

        if A[i][-1]:

            return None

    # ------------------------------
    # Free variables
    # ------------------------------

    free_cols = []

    for c in range(cols):

        if c not in pivot_cols:

            free_cols.append(c)

    # ------------------------------
    # Random free variables
    # ------------------------------

    x = [0] * cols

    for c in free_cols:

        x[c] = random.randint(0, 1)

    # ------------------------------
    # Back substitution
    # ------------------------------

    for i in range(len(pivot_cols) - 1, -1, -1):

        col = pivot_cols[i]

        value = A[i][-1]

        for j in range(col + 1, cols):

            value ^= (A[i][j] & x[j])

        x[col] = value

    return x


# ==========================================================
# Generate share vectors
# ==========================================================

all_share_vectors = []

print("\n========================================")
print("SHARE VECTORS")
print("========================================")

for bit_index, secret in enumerate(secret_bits, start=1):

    print(f"\nSecret Bit {bit_index} = {secret}")

    vectors = []

    for block_index, M in enumerate(matrices, start=1):

        v = random_affine_solution(M, secret)

        if v is None:

            print(f"Γ_{block_index} : No affine solution")

            vectors.append(None)

        else:

            vectors.append(v)

            print(f"Γ_{block_index} : {v}")

    all_share_vectors.append(vectors)




# ==========================================================
# Distribute share coordinates to participants
# ==========================================================

participant_shares = {}

# Initialize all participants

all_participants = set()

for plist in participant_lists:

    all_participants.update(plist)

for p in all_participants:

    participant_shares[p] = []

# ----------------------------------------------------------
# For every secret bit
# ----------------------------------------------------------

for vectors in all_share_vectors:

    # vectors contains one share vector for every partition block

    for plist, vec in zip(participant_lists, vectors):

        if vec is None:
            continue

        # Assign one coordinate to each participant
        for participant, coordinate in zip(plist, vec):

            participant_shares[participant].append(coordinate)

# ==========================================================
# Print participant shares
# ==========================================================

print("\n========================================")
print("FINAL PARTICIPANT SHARES")
print("========================================\n")

for p in sorted(participant_shares):

    share = "".join(map(str, participant_shares[p]))

    print(f"P{p} : {share}")

# ==========================================================
# Detailed shares
# ==========================================================

print("\n========================================")
print("DETAILED SHARE COORDINATES")
print("========================================\n")

for p in sorted(participant_shares):

    print(f"P{p} : {participant_shares[p]}")




# ==========================================================
# SUMMARY OF SHARE GENERATION
# ==========================================================

print("\n")
print("=" * 70)
print("SUMMARY OF SHARE GENERATION")
print("=" * 70)

for bit_index, vectors in enumerate(all_share_vectors, start=1):

    print("\n----------------------------------------")
    print(f"Secret Bit {bit_index} = {secret_bits[bit_index-1]}")
    print("----------------------------------------")

    for block_index, (plist, vec) in enumerate(
            zip(participant_lists, vectors), start=1):

        print(f"\nΓ_{block_index}")

        if vec is None:

            print("No affine solution exists.")

            continue

        print("Participants :", plist)

        print("Share Vector :", vec)

        print("Assignments")

        for participant, coordinate in zip(plist, vec):

            print(f"   P{participant}  <--  {coordinate}")

# ==========================================================
# SHARE LENGTHS
# ==========================================================

print("\n")
print("=" * 70)
print("SHARE LENGTHS")
print("=" * 70)

for p in sorted(participant_shares):

    print(
        f"P{p} : "
        f"{len(participant_shares[p])} bits"
    )

# ==========================================================
# PARTICIPANT SHARES PARTITION-WISE
# ==========================================================

print("\n")
print("=" * 70)
print("PARTITION-WISE PARTICIPANT SHARES")
print("=" * 70)

for block_index in range(num_partitions):

    print("\n")
    print("-" * 60)
    print(f"Γ_{block_index+1}")
    print("-" * 60)

    plist = participant_lists[block_index]

    # Header
    header = "Participant".ljust(15)

    for j in range(L):
        header += f"s{j+1}".center(6)

    print(header)
    print("-" * len(header))

    # Each participant
    for idx, participant in enumerate(plist):

        row = f"P{participant}".ljust(15)

        for bit in range(L):

            vec = all_share_vectors[bit][block_index]

            if vec is None:
                row += "-".center(6)
            else:
                row += str(vec[idx]).center(6)

        print(row)

# ==========================================================
# TOTAL SHARE STORAGE
# ==========================================================

total_bits = 0

for p in participant_shares:

    total_bits += len(participant_shares[p])

print("\n")
print("=" * 70)
print("STATISTICS")
print("=" * 70)

print("Secret Length           :", L)

print("Number of Participants  :", len(participant_shares))

print("Bitwise Encoding Complexity  :",
      round(total_bits / L, 2))

print("=" * 70)

Enter number of partition blocks: 2

Enter Γ_1 exactly like
['{1,2}','{2,3,4}']
Input : ['{1,4}', '{2,3}']

Enter Γ_2 exactly like
['{1,2}','{2,3,4}']
Input : ['{1,2}', '{1,3}', '{2,4}']

RECONSTRUCTION MATRICES

Γ_1
Participants : [1, 2, 3, 4]
[1, 0, 0, 1]
[0, 1, 1, 0]
Rank = 2

Γ_2
Participants : [1, 2, 3, 4]
[1, 1, 0, 0]
[1, 0, 1, 0]
[0, 1, 0, 1]
Rank = 3


Enter secret length : 5
Enter 5 bits separated by spaces
Secret : 1 0 1 0 1

SHARE VECTORS

Secret Bit 1 = 1
Γ_1 : [0, 0, 1, 1]
Γ_2 : [1, 0, 0, 1]

Secret Bit 2 = 0
Γ_1 : [0, 0, 0, 0]
Γ_2 : [0, 0, 0, 0]

Secret Bit 3 = 1
Γ_1 : [0, 1, 0, 1]
Γ_2 : [0, 1, 1, 0]

Secret Bit 4 = 0
Γ_1 : [1, 1, 1, 1]
Γ_2 : [1, 1, 1, 1]

Secret Bit 5 = 1
Γ_1 : [1, 0, 1, 0]
Γ_2 : [0, 1, 1, 0]

FINAL PARTICIPANT SHARES

P1 : 0100001110
P2 : 0000111101
P3 : 1000011111
P4 : 1100101100

DETAILED SHARE COORDINATES

P1 : [0, 1, 0, 0, 0, 0, 1, 1, 1, 0]
P2 : [0, 0, 0, 0, 1, 1, 1, 1, 0, 1]
P3 : [1, 0, 0, 0, 0, 1, 1, 1, 1, 1]
P4 : [1, 1, 0, 0, 1, 0, 1, 1, 0, 0]




## $R_S:$ Secret Reconstruction

In [ ]:
from itertools import combinations

# ==========================================================
# SECRET RECONSTRUCTION
# ==========================================================

print("\n")
print("=" * 70)
print("SECRET RECONSTRUCTION")
print("=" * 70)

# ----------------------------------------------------------
# Read Gamma_0
# ----------------------------------------------------------

m = int(input("\nEnter number of minimal qualified sets in Γ0 : "))

Gamma0 = []

print("\nEnter Γ0 sets like {1,2}")

for i in range(m):

    Gamma0.append(parse_set(input(f"A{i+1} : ")))

# ----------------------------------------------------------
# Reconstruction set
# ----------------------------------------------------------

print("\nEnter reconstruction set")

R = parse_set(input("Reconstruction Set : "))

qualified_set = None

for A in Gamma0:

    if A.issubset(R):

        qualified_set = A

        break

if qualified_set is None:

    print("\nThe reconstruction set is FORBIDDEN.")

    exit()

print("\nQualified Set Found :", sorted(qualified_set))

found = False

# ==========================================================
# CASE 1 : Explicitly present in some Γ_i
# ==========================================================

for block_index, block in enumerate(partitions):

    if qualified_set in block:

        found = True

        print("\nAppears explicitly in Γ_{}".format(block_index+1))

        recovered=[]

        for bit in range(L):

            vec=all_share_vectors[bit][block_index]

            plist=participant_lists[block_index]

            value=0

            for p in qualified_set:

                value ^= vec[plist.index(p)]

            recovered.append(value)

        print("\nRecovered Secret :", recovered)

        print("Recovered Bit String :",
              "".join(map(str,recovered)))

        break

# ==========================================================
# CASE 2 : Derived set (Property B)
# ==========================================================

if not found:

    for block_index, block in enumerate(partitions):

        n=len(block)

        for k in range(3,n+1,2):

            for sub in combinations(block,k):

                count={}

                for A in sub:

                    for x in A:

                        count[x]=count.get(x,0)+1

                xor_support=set()

                for x in count:

                    if count[x]%2==1:

                        xor_support.add(x)

                if xor_support==qualified_set:

                    found=True

                    print("\nDerived in Γ_{}".format(block_index+1))

                    print("Generated by")

                    for A in sub:

                        print(sorted(A))

                    recovered=[]

                    for bit in range(L):

                        secret=0

                        vec=all_share_vectors[bit][block_index]

                        plist=participant_lists[block_index]

                        for A in sub:

                            value=0

                            for p in A:

                                value ^= vec[plist.index(p)]

                            secret ^= value

                        recovered.append(secret)

                    print("\nRecovered Secret :", recovered)

                    print("Recovered Bit String :",
                          "".join(map(str,recovered)))

                    break

            if found:
                break

        if found:
            break

if not found:

    print("\nQualified set exists in Γ0 but cannot be reconstructed from the supplied partition.")



SECRET RECONSTRUCTION

Enter number of minimal qualified sets in Γ0 : 6

Enter Γ0 sets like {1,2}
A1 : {1,2}
A2 : {1,3}
A3 : {1,4}
A4 : {2,3}
A5 : {2,4}
A6 : {3,4}

Enter reconstruction set
Reconstruction Set : {1,2,3}

Qualified Set Found : [1, 2]

Appears explicitly in Γ_2

Recovered Secret : [1, 0, 1, 0, 1]
Recovered Bit String : 10101
